# Deploy VSS via the Orchestrator MCP server and OpenClaw UI

This notebook is meant to run **on the GPU host itself** — a [Brev](https://brev.nvidia.com) instance or any other Linux GPU host — and assumes you have already completed the companion notebook **`deploy_nemoclaw.ipynb`** — i.e. the NemoClaw sandbox is up, the VSS policy is applied, the VSS skills and workspace docs are installed, and the OpenClaw UI is reachable. With the default HTTP path, this notebook starts the host-side `vss_orchestrator` MCP (no prior sandbox registration). HTTPS MCP registration in `deploy_nemoclaw.ipynb` §3.5 is only required when `ORCHESTRATOR_ENABLE_HTTPS` is `True` in both notebooks.

A few steps are Brev-specific — the notebook reads secure-link FQDNs from `BREV_ENVIRONMENT_CONTEXT_PATH` (default `/etc/brev/environment-context.json`) and prints the remote VSS UI link. On other platforms reach the OpenClaw UI over your own networking — use the SSH tunnel shown in `deploy_nemoclaw.ipynb` section 3.7.

It walks through the rest of the flow for the **Video Search and Summarization (VSS)** blueprint: prepare the host for local NIM-backed VSS profiles, start the host-side VSS Orchestrator MCP server, then deploy and manage VSS from the OpenClaw UI (opened in `deploy_nemoclaw.ipynb`) by chatting with the agent.

**Default launchable (Brev):**  
[video-search-and-summarization-blueprint](https://brev.nvidia.com/launchable/deploy/now?launchableID=env-2tYIjRXL4eMCbH9Az8mJC5WPAI4)

On Brev, if you use that launchable the VM is expected to already have the main prerequisites in place. Otherwise — a different launchable, or another platform — use the preflight cells below to confirm what is missing.

**Required prerequisites**

- Run this notebook with **Python 3.11 or newer**. The MCP helper uses Python 3.11 standard-library APIs.
- Complete **`deploy_nemoclaw.ipynb`** first (sandbox onboarded). Optional section 6 auto-detects the default sandbox from `nemoclaw list` when probing host reachability.
- Make sure the intended VSS checkout is the one resolved by `VSS_REPO_DIR`. By default this notebook uses `~/video-search-and-summarization`; set the `VSS_REPO_DIR` environment variable before launching Jupyter if your checkout lives elsewhere.

**What this notebook covers**

- Set required keys and VSS LLM/VLM options.
- Run preflight checks for the agent environment, MCP helper/config, and host prerequisites.
- Prepare the host for local NIM-backed VSS profiles: log Docker into `nvcr.io`, and complete host backend prerequisites (MCP requirements + Docker NVIDIA runtime).
- Start the host-side VSS Orchestrator MCP server, then deploy and manage VSS from the OpenClaw UI (opened in `deploy_nemoclaw.ipynb`) by asking the agent to use the `vss_orchestrator__*` MCP tools. Teardown of a deployed VSS stack is also done from the UI (the agent invokes `docker_down`).
- *(Optional)* Verify sandbox-to-host reachability through `host.openshell.internal`.

**Security:** prefer `NVIDIA_API_KEY` and `NGC_CLI_API_KEY` from environment variables or your platform's secret store (e.g. Brev secrets). Do **not** commit notebook outputs that contain credentials or live access tokens.


## 1. Settings

Configure the credentials and hardware profile every VSS deployment needs, after completing `deploy_nemoclaw.ipynb`.

<span style="color:red"><strong>Important:</strong> set <code>NGC_CLI_API_KEY</code> below, and <code>NVIDIA_API_KEY</code> if you use NVIDIA-hosted VSS LLM/VLM endpoints. Credentials can also come from the environment or your platform's secret store (e.g. Brev secrets).</span>

### 1.1 Required settings

These apply to every deployment:

- `NGC_CLI_API_KEY` — NVIDIA legacy API key used to pull VSS container artifacts from `nvcr.io`.
- `NVIDIA_API_KEY` — build.nvidia.com key (`nvapi-...`); used by NVIDIA-hosted VSS LLM/VLM endpoints and passed to the MCP server.
- `HARDWARE_PROFILE` — selects the hardware-specific overlay.

In [ ]:
# ================== Required settings (always set) ==================
NGC_CLI_API_KEY = ""               # NVIDIA Legacy API key — used to pull VSS artifacts from nvcr.io
NVIDIA_API_KEY = ""                # build.nvidia.com key (nvapi-...) — used by NVIDIA-hosted VSS LLM/VLM
HARDWARE_PROFILE = "RTXPRO6000BW"  # H100 | GB300 | L40S | RTXPRO4500BW | RTXPRO6000BW | DGX-SPARK | IGX-THOR | AGX-THOR | OTHER


### 1.2 Advanced settings (defaults — usually leave alone)

VSS LLM/VLM overrides, GPU device ids, agent-adapter settings, and MCP server plumbing. `VSS_UI_PORT` is derived (default `7777`; override via `VSS_UI_PORT`). `BREV_ENVIRONMENT_CONTEXT_PATH` is derived (default `/etc/brev/environment-context.json`) — override via that env var only if the context file lives elsewhere. `EXTERNAL_IP` falls back to `hostname -I` when blank. Set `LLM_ENDPOINT_URL` / `VLM_ENDPOINT_URL` non-empty only to override the profile defaults with remote endpoints.

Structured HITL is disabled by default for every backend. Set `HITL_ENABLED=True` only for a legacy `vss-agent` deployment where both the agent and UI should pause for inline interaction responses. To route the deployed VSS UI chat surfaces to an external harness, set `VSS_AGENT_ADAPTER_ENABLED=True` and provide a Docker-reachable `VSS_AGENT_BACKEND_URL`; this notebook deliberately does not infer one because a loopback-only harness forward is unreachable from the UI containers. Set `VSS_AGENT_BACKEND_PROTOCOL` and `VSS_AGENT_BACKEND_PATH` for the harness protocol, and provide the gateway token through `VSS_AGENT_BACKEND_TOKEN` (prefer an environment secret). The notebook forces HITL back off because OpenClaw follow-up questions use ordinary completed chat turns.

Orchestrator MCP TLS: `ORCHESTRATOR_ENABLE_HTTPS` (`True`/`False`) selects the scheme the MCP server listens on. When enabled, leave `ORCHESTRATOR_CERTFILE` and `ORCHESTRATOR_KEYFILE` blank to auto-generate a self-signed pair under `.orchestrator-artifacts/`, or point them at your own PEM files (both must exist — a half-populated pair is an error rather than a silent regeneration). Whichever you use is validated in preflight. Override via the `ORCHESTRATOR_ENABLE_HTTPS` env var (`true`/`false`).

> Keep this toggle in sync with `ORCHESTRATOR_ENABLE_HTTPS` in `deploy_nemoclaw.ipynb`. With the default `False`, this notebook serves HTTP and the agent reaches it without a sandbox `mcp add`. Set `True` in both notebooks only when you want HTTPS; then `deploy_nemoclaw.ipynb` §3.5 registers that HTTPS MCP URL with the sandbox, and the scheme must match what this notebook serves.


In [ ]:
import os
import subprocess
from pathlib import Path


# ================== VSS settings ==================
LLM_NAME = ""
LLM_ENDPOINT_URL = ""
LLM_MODEL_TYPE = ""
LLM_ENABLE_THINKING = ""
OPENAI_API_KEY = ""            # blank => fall back to NVIDIA_API_KEY (typical for integrate.api.nvidia.com)

# Remote VLM — set VLM_ENDPOINT_URL non-empty to force VLM_MODE=remote in generated.env
VLM_NAME = ""
VLM_ENDPOINT_URL = ""
VLM_MODEL_TYPE = ""

LLM_DEVICE_ID = "0"
VLM_DEVICE_ID = "1"
EXTERNAL_IP = ""                    # blank => resolve from `hostname -I`

# ================== VSS UI -> agent adapter ==================
VSS_AGENT_ADAPTER_ENABLED = False
VSS_AGENT_BACKEND_PROTOCOL = "openclaw-ws"
VSS_AGENT_BACKEND_URL = ""           # required when the adapter is enabled; must be Docker-reachable
VSS_AGENT_BACKEND_PATH = ""          # blank => protocol default (/ for OpenClaw)
VSS_AGENT_BACKEND_TOKEN = ""         # prefer the environment; never commit a live token
HITL_ENABLED = False                    # explicit opt-in for vss-agent; always false for an external adapter

# ================== VSS Orchestrator MCP TLS settings ==================
ORCHESTRATOR_ENABLE_HTTPS = False
# Leave blank to use <ARTIFACT_DIR>/orchestrator_mcp_{cert,key}.pem (set in Derived).
ORCHESTRATOR_CERTFILE = ""
ORCHESTRATOR_KEYFILE = ""
ORCHESTRATOR_SSL_SAN = "DNS:localhost,IP:127.0.0.1"  # Derived also adds the host alias and EXTERNAL_IP


# ================== Derived (no need to touch) ==================

HOME_DIR = Path.home().resolve()
NVIDIA_API_KEY = (NVIDIA_API_KEY or os.environ.get("NVIDIA_API_KEY", "")).strip()
NGC_CLI_API_KEY = (NGC_CLI_API_KEY or os.environ.get("NGC_CLI_API_KEY", "")).strip()
HARDWARE_PROFILE = (HARDWARE_PROFILE or os.environ.get("HARDWARE_PROFILE", "RTXPRO6000BW")).strip()
EXTERNAL_IP = (EXTERNAL_IP or os.environ.get("EXTERNAL_IP", "")).strip()
if not EXTERNAL_IP:
    try:
        EXTERNAL_IP = subprocess.check_output(["hostname", "-I"], text=True).split()[0]
    except (subprocess.SubprocessError, IndexError):
        EXTERNAL_IP = ""
# HAProxy / VSS UI secure-link port. Override via VSS_UI_PORT if needed.
VSS_UI_PORT = int(os.environ.get("VSS_UI_PORT", "7777") or "7777")
# Brev context JSON (secure-link FQDNs + environment id). Override via env if needed.
BREV_ENVIRONMENT_CONTEXT_PATH = os.environ.get("BREV_ENVIRONMENT_CONTEXT_PATH", "/etc/brev/environment-context.json").strip()
os.environ["BREV_ENVIRONMENT_CONTEXT_PATH"] = BREV_ENVIRONMENT_CONTEXT_PATH
VSS_REPO_DIR = Path(os.environ.get("VSS_REPO_DIR", HOME_DIR / "video-search-and-summarization")).resolve()
DEPLOY_SCRIPTS_DIR = VSS_REPO_DIR / "deploy" / "docker" / "scripts"
AGENT_DIR = VSS_REPO_DIR / "services" / "agent"
ORCHESTRATOR_MCP_VENV_DIR = AGENT_DIR / ".venv"
ORCHESTRATOR_MCP_PYTHON_VERSION = "3.13"
MCP_CONFIG_PATH = DEPLOY_SCRIPTS_DIR / "vss_orchestrator_mcp_config.yml"
ORCHESTRATOR_MCP_HELPER_PATH = DEPLOY_SCRIPTS_DIR / "orchestrator_mcp_helper.py"
BREV_UTIL_PATH = AGENT_DIR / "packages" / "vss_agents" / "src" / "vss_agents" / "orchestrator" / "brev_util.py"
ARTIFACT_DIR = VSS_REPO_DIR / ".orchestrator-artifacts"
LOG_PATH = ARTIFACT_DIR / "vss_orchestrator_mcp.log"
UV_BIN_DIR = HOME_DIR / ".local" / "bin" / "uv"
VSS_AGENT_ADAPTER_ENABLED = (
    os.environ.get("VSS_AGENT_ADAPTER_ENABLED", str(VSS_AGENT_ADAPTER_ENABLED)).strip().lower()
    in ("1", "true", "yes", "on")
)
VSS_AGENT_BACKEND_PROTOCOL = (
    os.environ.get("VSS_AGENT_BACKEND_PROTOCOL", VSS_AGENT_BACKEND_PROTOCOL).strip().lower()
    or "openclaw-ws"
)
VSS_AGENT_BACKEND_URL = (
    os.environ.get("VSS_AGENT_BACKEND_URL", VSS_AGENT_BACKEND_URL).strip()
)
VSS_AGENT_BACKEND_PATH = (
    os.environ.get("VSS_AGENT_BACKEND_PATH", VSS_AGENT_BACKEND_PATH).strip()
)
VSS_AGENT_BACKEND_TOKEN = (
    os.environ.get("VSS_AGENT_BACKEND_TOKEN", VSS_AGENT_BACKEND_TOKEN).strip()
)
if VSS_AGENT_ADAPTER_ENABLED and not VSS_AGENT_BACKEND_URL:
    raise ValueError(
        "VSS_AGENT_BACKEND_URL is required when VSS_AGENT_ADAPTER_ENABLED=true; "
        "set it to a backend address reachable from the VSS UI containers"
    )
_hitl_enabled_raw = os.environ.get("HITL_ENABLED", str(HITL_ENABLED)).strip().lower()
if _hitl_enabled_raw not in ("1", "true", "yes", "on", "0", "false", "no", "off"):
    raise ValueError("HITL_ENABLED must be true or false")
HITL_ENABLED = _hitl_enabled_raw in ("1", "true", "yes", "on")
if VSS_AGENT_ADAPTER_ENABLED:
    HITL_ENABLED = False
LLM_DEVICE_ID = str(LLM_DEVICE_ID or os.environ.get("LLM_DEVICE_ID", "")).strip()
VLM_DEVICE_ID = str(VLM_DEVICE_ID or os.environ.get("VLM_DEVICE_ID", "")).strip()
# GB300 shares one GPU across every service, but unlike the edge boards it
# gets no device pinning from the orchestrator, so it happens here — blanks
# included, since search and alerts pin non-zero devices. A GB300 at a
# non-zero index needs the ids passed as docker_generate env_overrides.
if HARDWARE_PROFILE == "GB300" and (LLM_DEVICE_ID, VLM_DEVICE_ID) != ("0", "0"):
    print("[INFO] GB300 shares one GPU: collapsing LLM_DEVICE_ID/VLM_DEVICE_ID onto device 0")
    LLM_DEVICE_ID = VLM_DEVICE_ID = "0"
LLM_NAME = (LLM_NAME or os.environ.get("LLM_NAME", "")).strip()
LLM_ENDPOINT_URL = (LLM_ENDPOINT_URL or os.environ.get("LLM_ENDPOINT_URL", "")).strip()
LLM_MODEL_TYPE = (LLM_MODEL_TYPE or os.environ.get("LLM_MODEL_TYPE", "")).strip()
LLM_ENABLE_THINKING = (LLM_ENABLE_THINKING or os.environ.get("LLM_ENABLE_THINKING", "")).strip()
OPENAI_API_KEY = (OPENAI_API_KEY or os.environ.get("OPENAI_API_KEY", "")).strip()
VLM_NAME = (VLM_NAME or os.environ.get("VLM_NAME", "")).strip()
VLM_ENDPOINT_URL = (VLM_ENDPOINT_URL or os.environ.get("VLM_ENDPOINT_URL", "")).strip()
VLM_MODEL_TYPE = (VLM_MODEL_TYPE or os.environ.get("VLM_MODEL_TYPE", "")).strip()
MCP_HOST = os.environ.get("VSS_ORCHESTRATOR_MCP_HOST", "0.0.0.0").strip()
MCP_PORT = int(os.environ.get("VSS_ORCHESTRATOR_MCP_PORT", "9988"))
HOST_INTERNAL_ALIAS = os.environ.get("HOST_INTERNAL_ALIAS", "host.openshell.internal").strip()
# Must match ORCHESTRATOR_ENABLE_HTTPS in deploy_nemoclaw.ipynb.
ORCHESTRATOR_ENABLE_HTTPS = (os.environ.get("ORCHESTRATOR_ENABLE_HTTPS", str(ORCHESTRATOR_ENABLE_HTTPS)).strip().lower() == "true")
ORCHESTRATOR_CERTFILE = (ORCHESTRATOR_CERTFILE or os.environ.get("ORCHESTRATOR_CERTFILE", str(ARTIFACT_DIR / "orchestrator_mcp_cert.pem"))).strip()
ORCHESTRATOR_KEYFILE = (ORCHESTRATOR_KEYFILE or os.environ.get("ORCHESTRATOR_KEYFILE", str(ARTIFACT_DIR / "orchestrator_mcp_key.pem"))).strip()
ORCHESTRATOR_SSL_SAN = (ORCHESTRATOR_SSL_SAN or os.environ.get("ORCHESTRATOR_SSL_SAN", "")).strip()
MCP_SCHEME = "https" if ORCHESTRATOR_ENABLE_HTTPS else "http"
# MCP_HOST is the bind address; dial loopback when it is bind-all (not a connectable host).
MCP_URL = (
    f"{MCP_SCHEME}://127.0.0.1:{MCP_PORT}/mcp"
    if MCP_HOST in ("0.0.0.0", "::", "")
    else f"{MCP_SCHEME}://{MCP_HOST}:{MCP_PORT}/mcp"
)
if f"DNS:{HOST_INTERNAL_ALIAS}" not in ORCHESTRATOR_SSL_SAN:
    ORCHESTRATOR_SSL_SAN = (f"DNS:{HOST_INTERNAL_ALIAS},{ORCHESTRATOR_SSL_SAN}" if ORCHESTRATOR_SSL_SAN else f"DNS:{HOST_INTERNAL_ALIAS}")
if EXTERNAL_IP and f"IP:{EXTERNAL_IP}" not in ORCHESTRATOR_SSL_SAN:
    ORCHESTRATOR_SSL_SAN = f"{ORCHESTRATOR_SSL_SAN},IP:{EXTERNAL_IP}" if ORCHESTRATOR_SSL_SAN else f"IP:{EXTERNAL_IP}"
print("\nHOME_DIR:", HOME_DIR)
print("VSS_REPO_DIR:", VSS_REPO_DIR)
print("AGENT_DIR:", AGENT_DIR)
print("MCP_CONFIG_PATH:", MCP_CONFIG_PATH)
print("ORCHESTRATOR_MCP_HELPER_PATH:", ORCHESTRATOR_MCP_HELPER_PATH)
print("BREV_UTIL_PATH:", BREV_UTIL_PATH)
print("ARTIFACT_DIR:", ARTIFACT_DIR)
print("LOG_PATH:", LOG_PATH)
print("HARDWARE_PROFILE:", HARDWARE_PROFILE)
print("EXTERNAL_IP:", EXTERNAL_IP or "(unresolved)")
print("VSS_UI_PORT:", VSS_UI_PORT)
print("VSS_AGENT_ADAPTER_ENABLED:", VSS_AGENT_ADAPTER_ENABLED)
print("HITL_ENABLED:", HITL_ENABLED)
if VSS_AGENT_ADAPTER_ENABLED:
    print("VSS_AGENT_BACKEND_PROTOCOL:", VSS_AGENT_BACKEND_PROTOCOL)
    print("VSS_AGENT_BACKEND_URL:", VSS_AGENT_BACKEND_URL)
    print("VSS_AGENT_BACKEND_PATH:", VSS_AGENT_BACKEND_PATH or "(protocol default)")
    print("VSS_AGENT_BACKEND_TOKEN set:", bool(VSS_AGENT_BACKEND_TOKEN))
print("BREV_ENVIRONMENT_CONTEXT_PATH:", BREV_ENVIRONMENT_CONTEXT_PATH)
print("LLM_DEVICE_ID:", LLM_DEVICE_ID or "(profile default)")
print("VLM_DEVICE_ID:", VLM_DEVICE_ID or "(profile default)")
print("HOST_INTERNAL_ALIAS:", HOST_INTERNAL_ALIAS)
print("MCP_URL:", MCP_URL)
print("ORCHESTRATOR_ENABLE_HTTPS:", ORCHESTRATOR_ENABLE_HTTPS)
if ORCHESTRATOR_ENABLE_HTTPS:
    print("ORCHESTRATOR_CERTFILE:", ORCHESTRATOR_CERTFILE or "(unset)")
    print("ORCHESTRATOR_KEYFILE:", ORCHESTRATOR_KEYFILE or "(unset)")
    print("ORCHESTRATOR_SSL_SAN:", ORCHESTRATOR_SSL_SAN or "(unset)")
if LLM_ENDPOINT_URL:
    print("LLM_NAME:", LLM_NAME)
    print("LLM_ENDPOINT_URL:", LLM_ENDPOINT_URL)
    print("LLM_MODEL_TYPE:", LLM_MODEL_TYPE)
print("LLM_ENABLE_THINKING:", LLM_ENABLE_THINKING)
if VLM_ENDPOINT_URL:
    print("VLM_NAME:", VLM_NAME)
    print("VLM_ENDPOINT_URL:", VLM_ENDPOINT_URL)
    print("VLM_MODEL_TYPE:", VLM_MODEL_TYPE)
print("NGC_CLI_API_KEY set:", bool(NGC_CLI_API_KEY))
print("NVIDIA_API_KEY set:", bool(NVIDIA_API_KEY))


## 2. Preflight

Run the next cell to confirm the expected keys, files, and commands are present on the host, load the MCP helper module, and (on Brev) confirm the VSS UI secure-link FQDN from `BREV_ENVIRONMENT_CONTEXT_PATH`.

The cell prints the whole checklist and then fails if any required entry is missing, so you see every problem at once. `NGC_CLI_API_KEY` is one of them — nothing later re-checks it before the containers try to use it.

> Docker version pinning is handled in `deploy_nemoclaw.ipynb` (it must run before the OpenClaw sandbox comes up). This notebook assumes Docker is already pinned to the tested range.

In [ ]:
import importlib.util
import os
import shutil
from pathlib import Path

RED = "\033[31m"
RESET = "\033[0m"
GREEN = "\033[32m"
YELLOW = "\033[33m"

helper_spec = importlib.util.spec_from_file_location("orchestrator_mcp_helper", ORCHESTRATOR_MCP_HELPER_PATH)
if helper_spec is None or helper_spec.loader is None:
    raise ImportError(f"Could not load MCP helper from {ORCHESTRATOR_MCP_HELPER_PATH}")
orchestrator_mcp_helper = importlib.util.module_from_spec(helper_spec)
helper_spec.loader.exec_module(orchestrator_mcp_helper)

brev_util_spec = importlib.util.spec_from_file_location("vss_brev_util", BREV_UTIL_PATH)
if brev_util_spec is None or brev_util_spec.loader is None:
    raise ImportError(f"Could not load brev_util from {BREV_UTIL_PATH}")
brev_util = importlib.util.module_from_spec(brev_util_spec)
brev_util_spec.loader.exec_module(brev_util)

OrchestratorTool = orchestrator_mcp_helper.OrchestratorTool
poll_compose_op = orchestrator_mcp_helper.poll_compose_op
require_success = orchestrator_mcp_helper.require_success
tool_call = orchestrator_mcp_helper.tool_call
host_gpu_device_ids = orchestrator_mcp_helper.gpu_device_ids
require_gpu_device = orchestrator_mcp_helper.require_gpu_device
supported_hardware_profiles = orchestrator_mcp_helper.supported_hardware_profiles
resolve_openshell_gateway_container = orchestrator_mcp_helper.resolve_openshell_gateway_container
brev_environment_id = brev_util.brev_environment_id
brev_secure_link_fqdn = brev_util.brev_secure_link_fqdn

if ORCHESTRATOR_ENABLE_HTTPS:
    _cert_path, _key_path = orchestrator_mcp_helper.ensure_mcp_tls_certs(
        ORCHESTRATOR_CERTFILE,
        ORCHESTRATOR_KEYFILE,
        san=ORCHESTRATOR_SSL_SAN,
    )
    ORCHESTRATOR_CERTFILE = str(_cert_path)
    ORCHESTRATOR_KEYFILE = str(_key_path)
    # Trust this cert for section 4's `nat mcp client` health check (HTTPS).
    os.environ["SSL_CERT_FILE"] = ORCHESTRATOR_CERTFILE

if not shutil.which("uv") and UV_BIN_DIR.exists():
    os.environ["PATH"] = f"{UV_BIN_DIR.parent}:{os.environ.get('PATH', '')}"

_brev_env_id = brev_environment_id()
_vss_ui_fqdn = brev_secure_link_fqdn(VSS_UI_PORT)
print("BREV_ENV_ID:", _brev_env_id or "(unset)")
print("VSS UI FQDN (port", f"{VSS_UI_PORT}):", _vss_ui_fqdn or "(not in context file)")
print("VSS UI URL:", f"https://{_vss_ui_fqdn}/" if _vss_ui_fqdn else "(unavailable)")

# Fail fast on GPU device ids this host does not have (e.g. 0 and 1 on a single-GPU box).
gpu_indices, gpu_device_ids = host_gpu_device_ids()
for label, device_id, endpoint_url in (
    ("LLM_DEVICE_ID", LLM_DEVICE_ID, LLM_ENDPOINT_URL),
    ("VLM_DEVICE_ID", VLM_DEVICE_ID, VLM_ENDPOINT_URL),
):
    # A remote endpoint runs the model off-host, so its device id is never used.
    if endpoint_url:
        continue
    require_gpu_device(
        label,
        device_id,
        remedy="Fix it in section 1.2, or leave it blank for the profile default.",
        known_device_ids=gpu_device_ids,
    )
print("GPU indices:", ", ".join(gpu_indices))

SUPPORTED_HARDWARE_PROFILES = supported_hardware_profiles(MCP_CONFIG_PATH) if MCP_CONFIG_PATH.is_file() else ()

required_checks = {
    "NGC_CLI_API_KEY set": bool(NGC_CLI_API_KEY),
    "HARDWARE_PROFILE supported": HARDWARE_PROFILE in SUPPORTED_HARDWARE_PROFILES,
    "EXTERNAL_IP resolved": bool(EXTERNAL_IP),
    "services/agent/": AGENT_DIR.is_dir(),
    "services/agent/pyproject.toml": (AGENT_DIR / "pyproject.toml").is_file(),
    "vss_orchestrator_mcp_config.yml": MCP_CONFIG_PATH.is_file(),
    "orchestrator_mcp_helper.py": ORCHESTRATOR_MCP_HELPER_PATH.is_file(),
    "brev_util.py": BREV_UTIL_PATH.is_file(),
    # host commands
    "docker": shutil.which("docker") is not None,
    "python3": shutil.which("python3") is not None,
    "curl": shutil.which("curl") is not None,
    "uv": shutil.which("uv") is not None,
}

required_checks["ORCHESTRATOR_ENABLE_HTTPS is bool"] = isinstance(ORCHESTRATOR_ENABLE_HTTPS, bool)
if ORCHESTRATOR_ENABLE_HTTPS:
    required_checks["ORCHESTRATOR_CERTFILE set and exists"] = bool(ORCHESTRATOR_CERTFILE) and Path(ORCHESTRATOR_CERTFILE).is_file()
    required_checks["ORCHESTRATOR_KEYFILE set and exists"] = bool(ORCHESTRATOR_KEYFILE) and Path(ORCHESTRATOR_KEYFILE).is_file()

optional_checks = {
    "orchestrator MCP venv": ORCHESTRATOR_MCP_VENV_DIR.is_dir(),
}

for label, ok in required_checks.items():
    status = "OK " if ok else "NO "
    color = GREEN if ok else RED
    print(f"{color}{status}{RESET} {label}")

for label, ok in optional_checks.items():
    status = "OK " if ok else "-- "
    color = GREEN if ok else YELLOW
    print(f"{color}{status}{RESET} {label} (optional)")

if SUPPORTED_HARDWARE_PROFILES and HARDWARE_PROFILE not in SUPPORTED_HARDWARE_PROFILES:
    print(
        f"{YELLOW}HARDWARE_PROFILE={HARDWARE_PROFILE!r} is not one of "
        f"{', '.join(SUPPORTED_HARDWARE_PROFILES)} — fix it in section 1.1.{RESET}"
    )

# Raise only after the whole checklist has printed: the point of the list is to show
# every problem at once rather than stopping at the first.
_failed_checks = [label for label, ok in required_checks.items() if not ok]
if _failed_checks:
    raise RuntimeError(
        "Preflight failed — fix the following, then re-run this cell: "
        + ", ".join(_failed_checks)
    )


## 3. Prepare the host for local NIM-backed VSS profiles

If you plan to deploy local NIM-backed VSS profiles through the orchestrator tools, complete the next two pre-steps on the host first:

1. authenticate Docker to `nvcr.io`, and
2. prepare the `services/agent/` Python environment.

The NGC CLI is **not** needed on this host, so there is no step to install it. Every NGC *model* download happens inside a container — the perception container bootstraps its own `ngc` during `ds-start.sh` phase 0, and each download passes `--org` explicitly, so no `~/.ngc/config` is read either. All the host supplies is `NGC_CLI_API_KEY`: section 2 checks it is set, section 3.1 below uses it to authenticate Docker for *image* pulls, and section 4 passes it to the MCP server, which forwards it to the containers.

### 3.1 Docker login to `nvcr.io`

If you plan to deploy local NIM-backed VSS profiles, authenticate Docker to the NVIDIA Container Registry before using the orchestrator deployment tools.

This pre-step runs `docker login nvcr.io` with `NGC_CLI_API_KEY`.

In [ ]:
import subprocess

if not NGC_CLI_API_KEY:
    raise RuntimeError("NGC_CLI_API_KEY is not set. Export it before running this cell.")

login_result = subprocess.run(
    [
        "docker",
        "login",
        "nvcr.io",
        "--username",
        "$oauthtoken",
        "--password",
        NGC_CLI_API_KEY,
    ],
    capture_output=True,
    text=True,
)
if login_result.returncode != 0:
    raise RuntimeError(f"Docker login to nvcr.io failed\n{login_result.stderr}")

print("Docker login to nvcr.io: OK")

### 3.2 Install VSS host backend prerequisites

Two host-side preparations for local NIM-backed VSS profiles and the orchestrator MCP server:

- **MCP requirements** — install `uv`, create the `services/agent/` virtualenv, and sync the Python packages needed for `uv run nat mcp …`.
- **VSS backend deps** — if Docker does not list the `nvidia` runtime (or a compose-style smoke test fails), run `nvidia-ctk runtime configure` and restart Docker, then verify with `runtime: nvidia`.

> Run both subsections below before starting the orchestrator MCP server.

#### 3.2.1 MCP requirements

The orchestrator MCP server runs from `services/agent/` via `uv run nat mcp ...`. This cell installs `libcairo2-dev`, `pkg-config`, and `python3-dev` if any are missing, auto-installs `uv` if missing, creates `services/agent/.venv` when needed, and runs `uv sync --no-dev --extra agent` (the `agent` extra provides `nvidia-nat[mcp]`, which registers the `nat mcp` command).

In [ ]:
import os
from pathlib import Path
import shutil
import subprocess

REQUIRED_MCP_APT_PACKAGES = ("libcairo2-dev", "pkg-config", "python3-dev")


def apt_package_installed(package: str) -> bool:
    result = subprocess.run(
        ["dpkg-query", "-W", "-f=${Status}", package],
        capture_output=True,
        text=True,
    )
    return result.returncode == 0 and result.stdout.strip() == "install ok installed"


def ensure_apt_packages(packages: tuple[str, ...]) -> None:
    missing = [package for package in packages if not apt_package_installed(package)]
    if not missing:
        print("Apt packages already installed:", ", ".join(packages))
        return

    print("Installing missing apt packages:", ", ".join(missing))
    subprocess.run(["sudo", "apt-get", "update", "-qq"], check=True)
    subprocess.run(
        ["sudo", "env", "DEBIAN_FRONTEND=noninteractive", "apt-get", "install", "-y", *missing],
        check=True,
    )


def ensure_uv_on_path() -> None:
    uv_bin_dir = Path.home() / ".local" / "bin"
    os.environ["PATH"] = f"{uv_bin_dir}:{os.environ.get('PATH', '')}"
    if shutil.which("uv") is None:
        print("Installing uv ...")
        installer = subprocess.run(
            ["curl", "-LsSf", "https://astral.sh/uv/install.sh"],
            check=True,
            capture_output=True,
            text=True,
        )
        subprocess.run(["sh"], input=installer.stdout, text=True, check=True)
        os.environ["PATH"] = f"{uv_bin_dir}:{os.environ.get('PATH', '')}"
    if shutil.which("uv") is None:
        raise RuntimeError("uv is not installed and auto-install failed.")


def uv_env_for_agent() -> dict[str, str]:
    env = os.environ.copy()
    # Do not inherit the notebook kernel venv; uv should use services/agent/.venv.
    env.pop("VIRTUAL_ENV", None)
    return env


def run_uv_sync() -> subprocess.CompletedProcess[str]:
    return subprocess.run(
        ["uv", "sync", "--no-dev", "--extra", "agent"],
        cwd=str(AGENT_DIR),
        env=uv_env_for_agent(),
        check=False,
        capture_output=True,
        text=True,
    )


ensure_apt_packages(REQUIRED_MCP_APT_PACKAGES)
ensure_uv_on_path()
if not ORCHESTRATOR_MCP_VENV_DIR.is_dir():
    print(f"Creating Python {ORCHESTRATOR_MCP_PYTHON_VERSION} venv in {ORCHESTRATOR_MCP_VENV_DIR} ...")
    subprocess.run(
        ["uv", "venv", "--python", ORCHESTRATOR_MCP_PYTHON_VERSION],
        cwd=str(AGENT_DIR),
        check=True,
    )

print("Installing orchestrator MCP dependencies ...")
sync_result = run_uv_sync()
if sync_result.returncode != 0:
    message = (
        "uv sync failed while preparing the orchestrator MCP environment."
        f"\nSTDOUT:\n{sync_result.stdout}"
        f"\nSTDERR:\n{sync_result.stderr}"
    )
    raise RuntimeError(message)

agent_env = uv_env_for_agent()
subprocess.run(
    ["uv", "run", "nat", "mcp", "--help"],
    cwd=str(AGENT_DIR),
    env=agent_env,
    check=True,
)
module_check = subprocess.run(
    [
        "uv",
        "run",
        "python",
        "-c",
        (
            "from importlib.metadata import entry_points; "
            "import importlib; "
            "eps = entry_points(group='nat.components'); "
            "mod = next(e.value for e in eps if e.name == 'vss_orchestrator_mcp'); "
            "importlib.import_module(mod); "
            "print(f'orchestrator MCP module OK ({mod})')"
        ),
    ],
    cwd=str(AGENT_DIR),
    env=agent_env,
    check=True,
    capture_output=True,
    text=True,
)
print(module_check.stdout.strip())
print(f"Orchestrator MCP venv ready in {ORCHESTRATOR_MCP_VENV_DIR}")

#### 3.2.2 VSS backend deps

Local NIM-backed VSS profiles need Docker to expose the NVIDIA runtime. This cell checks Docker runtime registration and a compose config using `runtime: nvidia`; if either check fails, it runs `nvidia-ctk runtime configure --runtime=docker`, restarts Docker, and verifies the runtime again.

In [ ]:
import json
import shutil
import subprocess
import time


def docker_runtimes() -> dict:
    result = subprocess.run(
        ["docker", "info", "--format", "{{json .Runtimes}}"],
        capture_output=True,
        text=True,
        check=False,
    )
    if result.returncode != 0:
        raise RuntimeError(f"docker info failed:\n{result.stderr}\n{result.stdout}")
    return json.loads(result.stdout or "{}")


def docker_has_nvidia_runtime() -> bool:
    return "nvidia" in docker_runtimes()


def compose_accepts_nvidia_runtime() -> bool:
    compose_yaml = '''
services:
  nvidia-runtime-smoke:
    image: busybox:latest
    runtime: nvidia
    command: ["true"]
'''.strip()
    result = subprocess.run(
        ["docker", "compose", "-f", "-", "config"],
        input=compose_yaml,
        capture_output=True,
        text=True,
        check=False,
    )
    if result.returncode != 0:
        print("Docker compose runtime smoke test failed:")
        print(result.stderr or result.stdout)
    return result.returncode == 0


def restart_docker() -> None:
    if shutil.which("systemctl"):
        subprocess.run(["sudo", "systemctl", "restart", "docker"], check=True)
    else:
        subprocess.run(["sudo", "service", "docker", "restart"], check=True)


def wait_for_docker(timeout_s: int = 60) -> None:
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        result = subprocess.run(["docker", "info"], capture_output=True, text=True, check=False)
        if result.returncode == 0:
            return
        time.sleep(2)
    raise RuntimeError("Docker did not become ready after restart.")


if shutil.which("docker") is None:
    raise RuntimeError("docker is not installed or not on PATH.")

runtime_ok = docker_has_nvidia_runtime()
compose_ok = compose_accepts_nvidia_runtime()
if not runtime_ok or not compose_ok:
    if shutil.which("nvidia-ctk") is None:
        raise RuntimeError(
            "Docker is missing the nvidia runtime and nvidia-ctk is not on PATH. "
            "Install NVIDIA Container Toolkit, then re-run this cell."
        )
    print("Configuring Docker NVIDIA runtime with nvidia-ctk ...")
    subprocess.run(["sudo", "nvidia-ctk", "runtime", "configure", "--runtime=docker"], check=True)
    restart_docker()
    wait_for_docker()

runtimes = docker_runtimes()
if "nvidia" not in runtimes:
    raise RuntimeError(f"Docker still does not list the nvidia runtime. Runtimes: {sorted(runtimes)}")
if not compose_accepts_nvidia_runtime():
    raise RuntimeError("Docker compose still does not accept runtime: nvidia.")

print("Docker NVIDIA runtime: OK")
print("Docker runtimes:", ", ".join(sorted(runtimes)))

## 4. Start the VSS Orchestrator MCP server

Start the host-side VSS Orchestrator MCP server — a host process listening on port `9988` that exposes the `vss_orchestrator__*` tools the agent uses to deploy and manage VSS. Once it is healthy, continue to **section 5** to verify the agent and drive the deployment from the Agent UI.

Run the next cell after the MCP requirements and VSS backend dependency cells to stop any previously recorded MCP server from this notebook session and start a fresh host-side listener on port `9988`.

The cell checks that the prerequisites completed (`uv`, orchestrator MCP venv, Docker, OpenShell, MCP config, and the TLS cert/key when HTTPS is enabled), then runs `uv run nat mcp serve` in the background and waits for the health check to pass. Logs are written to `.orchestrator-artifacts/vss_orchestrator_mcp.log`.

The scheme comes from `ORCHESTRATOR_ENABLE_HTTPS` (section 1.2). `vss_orchestrator_mcp_config.yml` pins `runner_class: vss_agents.orchestrator.mcp_ssl_worker.SSLMCPWorker`, and this cell passes the toggle plus the cert/key paths to the server subprocess; with the toggle off that worker behaves exactly like the stock NAT one.


In [ ]:
import os
import importlib.util
import shutil
import signal
import subprocess
import sys
import time
from pathlib import Path


helper_spec = importlib.util.spec_from_file_location("orchestrator_mcp_helper", ORCHESTRATOR_MCP_HELPER_PATH)
if helper_spec is None or helper_spec.loader is None:
    raise ImportError(f"Could not load MCP helper from {ORCHESTRATOR_MCP_HELPER_PATH}")
orchestrator_mcp_helper = importlib.util.module_from_spec(helper_spec)
helper_spec.loader.exec_module(orchestrator_mcp_helper)

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

serve_prereqs = {
    "uv": shutil.which("uv") is not None,
    "orchestrator MCP venv": ORCHESTRATOR_MCP_VENV_DIR.is_dir(),
    "docker": shutil.which("docker") is not None,
    "openshell": shutil.which("openshell") is not None,
    "agent dir": AGENT_DIR.is_dir(),
    "MCP config": MCP_CONFIG_PATH.is_file(),
}
if ORCHESTRATOR_ENABLE_HTTPS:
    serve_prereqs["MCP TLS cert"] = bool(ORCHESTRATOR_CERTFILE) and Path(ORCHESTRATOR_CERTFILE).is_file()
    serve_prereqs["MCP TLS key"] = bool(ORCHESTRATOR_KEYFILE) and Path(ORCHESTRATOR_KEYFILE).is_file()

for label, ok in serve_prereqs.items():
    if not ok:
        raise RuntimeError(f"Cannot start the MCP server because {label} is unavailable. Resolve this before proceeding.")


def _wait_for_mcp_health(process: subprocess.Popen, timeout_s: int = 60, interval_s: int = 3) -> None:
    deadline = time.time() + timeout_s
    last_error = "health check did not run"
    while time.time() < deadline:
        return_code = process.poll()
        if return_code is not None:
            raise RuntimeError(f"MCP server exited before becoming healthy with exit code {return_code}: {last_error}")
        try:
            healthy, message = orchestrator_mcp_helper.check_mcp_health(MCP_URL, AGENT_DIR)
        except subprocess.TimeoutExpired:
            healthy, message = False, "health command timed out"
        last_error = message
        if healthy:
            print(f"MCP health check passed: {message}")
            return
        time.sleep(interval_s)

    raise RuntimeError(f"MCP server did not become healthy within {timeout_s}s: {last_error}")


existing_pid = globals().get("VSS_ORCHESTRATOR_MCP_PID")
if existing_pid:
    try:
        os.kill(existing_pid, signal.SIGTERM)
        print(f"Stopped existing MCP server PID {existing_pid}")
        time.sleep(2)
    except ProcessLookupError:
        print(f"Recorded MCP server PID {existing_pid} is no longer running")

env = os.environ.copy()
env.setdefault("PYTHONUNBUFFERED", "1")
# Context file is the source of truth for secure-link FQDNs inside the agent.
env["BREV_ENVIRONMENT_CONTEXT_PATH"] = BREV_ENVIRONMENT_CONTEXT_PATH
env["PROXY_PORT"] = str(VSS_UI_PORT)
if NGC_CLI_API_KEY:
    env["NGC_CLI_API_KEY"] = NGC_CLI_API_KEY
if NVIDIA_API_KEY:
    env["NVIDIA_API_KEY"] = NVIDIA_API_KEY
if HARDWARE_PROFILE:
    env["HARDWARE_PROFILE"] = HARDWARE_PROFILE
env["HITL_ENABLED"] = "true" if HITL_ENABLED else "false"
if VSS_AGENT_ADAPTER_ENABLED:
    env["VSS_AGENT_ADAPTER_ENABLED"] = "true"
    env["VSS_AGENT_BACKEND_PROTOCOL"] = VSS_AGENT_BACKEND_PROTOCOL
    env["VSS_AGENT_BACKEND_URL"] = VSS_AGENT_BACKEND_URL
    env["VSS_AGENT_BACKEND_PATH"] = VSS_AGENT_BACKEND_PATH
    if VSS_AGENT_BACKEND_TOKEN:
        env["VSS_AGENT_BACKEND_TOKEN"] = VSS_AGENT_BACKEND_TOKEN
env["EXTERNAL_IP"] = EXTERNAL_IP
if LLM_ENABLE_THINKING:
    env["LLM_ENABLE_THINKING"] = LLM_ENABLE_THINKING
if LLM_DEVICE_ID:
    env["LLM_DEVICE_ID"] = LLM_DEVICE_ID
if VLM_DEVICE_ID:
    env["VLM_DEVICE_ID"] = VLM_DEVICE_ID
if LLM_ENDPOINT_URL or VLM_ENDPOINT_URL:
    env["OPENAI_API_KEY"] = OPENAI_API_KEY or NVIDIA_API_KEY
if LLM_ENDPOINT_URL:
    env["LLM_ENDPOINT_URL"] = LLM_ENDPOINT_URL
    env["LLM_NAME"] = LLM_NAME
    env["LLM_MODEL_TYPE"] = LLM_MODEL_TYPE
if VLM_ENDPOINT_URL:
    env["VLM_NAME"] = VLM_NAME
    env["VLM_ENDPOINT_URL"] = VLM_ENDPOINT_URL
    env["VLM_MODEL_TYPE"] = VLM_MODEL_TYPE
env["ORCHESTRATOR_ENABLE_HTTPS"] = "true" if ORCHESTRATOR_ENABLE_HTTPS else "false"
if ORCHESTRATOR_ENABLE_HTTPS:
    env["ORCHESTRATOR_CERTFILE"] = str(ORCHESTRATOR_CERTFILE)
    env["ORCHESTRATOR_KEYFILE"] = str(ORCHESTRATOR_KEYFILE)
log_handle = LOG_PATH.open("w")
process = subprocess.Popen(
    [
        "uv",
        "run",
        "nat",
        "mcp",
        "serve",
        "--config_file",
        str(MCP_CONFIG_PATH),
        "--host",
        MCP_HOST,
        "--port",
        str(MCP_PORT),
    ],
    cwd=str(AGENT_DIR),
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True,
)
VSS_ORCHESTRATOR_MCP_PID = process.pid
print(f"Started MCP server with PID {VSS_ORCHESTRATOR_MCP_PID}")
_wait_for_mcp_health(process)
print("MCP log:", LOG_PATH)
print("MCP URL:", MCP_URL)

## 5. Verify the Agent and deploy VSS from the UI

With the MCP server running (**section 4**) and the **Agent UI** open (from `deploy_nemoclaw.ipynb` section 3.7), confirm the agent sees the VSS skills and the `vss_orchestrator__*` MCP tools, then drive the VSS deployment through chat. From here on, the agent — not this notebook — drives the deployment.

> The prompts and screenshots use **OpenClaw as an example** — run the equivalent checks in whichever harness you onboarded (`AGENT_RUNTIME`, e.g. OpenClaw or Hermes).

#### Step 1. Verify the VSS skills are imported

Open the **Skills** tab in the Agent UI and confirm the **VSS skills** are present.

![Agent UI skills screenshot (OpenClaw example)](./images/OpenClawUISkills.png)

#### Step 2. Verify the agent sees the `vss_orchestrator` MCP tools

These checks require the MCP server from **section 4** to be running on port `9988`. If any step fails, re-check that the MCP server is still up.

**Prompt A — show deployment tools:**

> *"Show me the deployment tools."*

The agent should summarize the VSS deployment tools, including the `vss_orchestrator__*` tools registered by the MCP server.

![Agent UI list tools screenshot (OpenClaw example)](./images/OpenClawUIListTools.png)

**Prompt B — query a tool (list profiles):**

> *"List the available VSS deployment profiles."*

The agent should invoke `vss_orchestrator__profiles` and return `base`, `search`, `alerts`, `lvs`.

![Agent UI list profiles screenshot (OpenClaw example)](./images/OpenClawListVSSProfiles.png)

**Prompt C — invoke a deployment:**

> *"Deploy the VSS `alerts` profile in `verification` mode."*

The agent should chain `vss_orchestrator__docker_generate` → `docker_up` → `docker_status` and stream progress back into the chat. It will ask before invoking the build, and surface a `docker_compose_id` you can reference later.

![Agent UI deploy VSS screenshot (OpenClaw example)](./images/OpenClawUIDeploy.png)

#### Available `vss_orchestrator__*` MCP tools

The MCP server exposes nine tools (all prefixed `vss_orchestrator__`):

| Tool | Purpose |
| --- | --- |
| `profiles` | List supported deployment profiles (`base`, `search`, `alerts`, `lvs`). |
| `prereqs` | Run Docker / GPU / NGC prerequisite checks on the host. |
| `docker_generate` | Resolve `.env` + compose YAML artifacts for the chosen profile. |
| `docker_read` | Fetch generated env/yaml by `docker_compose_id`. |
| `docker_up` | `docker compose up -d --build --quiet-pull` for the generated artifacts. |
| `docker_status` | Poll status/logs of the most recent `docker_up` / `docker_down` operation. |
| `docker_list` | List currently running container names. |
| `docker_logs` | Fetch docker logs for a given container name. |
| `docker_down` | `docker compose down -v --remove-orphans` to tear the deployment back down. |

#### Sample prompts to trigger them

You do **not** call these tools by name — the agent picks the right tool from your natural-language request and chains them in the correct order. Try prompts like:

| Sample prompt | Tool(s) invoked |
| --- | --- |
| *"List the available VSS deployment profiles."* | `profiles` |
| *"Check that my host meets the prerequisites for the `alerts` profile."* | `prereqs` |
| *"Deploy the VSS `alerts` profile in `verification` mode."* | `docker_generate` → `docker_up` → `docker_status` |
| *"Show me the status of the deployment."* | `docker_status` |
| *"List the running VSS containers."* | `docker_list` |
| *"Fetch the last 200 lines of logs from `vss-alert-bridge`."* | `docker_logs` |
| *"Tear down the VSS deployment."* | `docker_down` |

The agent will ask before destructive steps (e.g. `docker_down`) and stream progress back into the chat. If a tool call fails, paste the error message back to the agent and ask it to remediate — it has access to `docker_logs` and `docker_status` to diagnose.


## 6. [OPTIONAL] Verify host reachability from inside the sandbox

<span style="color:red"><strong>Important:</strong> make sure VSS is already deployed on the host before running this step.</span>

Run the next cell only after the agent has finished deploying VSS in **section 5**. It auto-detects the default sandbox from `nemoclaw list --json` (override via `NEMOCLAW_SANDBOX_NAME`), then runs `nemoclaw <sandbox> connect` and feeds a probe script into that real sandbox session.
Do **not** replace this with `docker exec`; that can use a different path and produce misleading results.

The `STATUS` column is intentionally simple:

- `REACHABLE` — an HTTP service answered on that port. `404` and non-policy `403` still count because they prove a service is listening, even if `GET /` is not a valid or authorized route.
- `NOT_REACHABLE` — the port is blocked by policy, no service is listening, DNS failed, or the request timed out. Check the `NOTE` column for the reason.


In [ ]:
import json
import os
import re
import subprocess

required_vars = ("HOST_INTERNAL_ALIAS",)
missing_vars = [name for name in required_vars if name not in globals()]
if missing_vars:
    raise RuntimeError("Required variables are missing: " + ", ".join(missing_vars))

TEST_PORTS = (3000, 8000, 9988, 30888, 5601, 6006, 9200, 8081, 31000, 9901, 38111, 38112)


def _nemoclaw_list_payload() -> dict:
    try:
        return json.loads(subprocess.check_output(["nemoclaw", "list", "--json"], text=True))
    except FileNotFoundError as exc:
        raise RuntimeError(
            "nemoclaw CLI not found; complete deploy_nemoclaw.ipynb first, "
            "or set NEMOCLAW_SANDBOX_NAME in the environment."
        ) from exc
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            f"`nemoclaw list --json` failed (exit {exc.returncode}); "
            "set NEMOCLAW_SANDBOX_NAME in the environment to override."
        ) from exc
    except json.JSONDecodeError as exc:
        raise RuntimeError(
            "`nemoclaw list --json` did not return valid JSON; "
            "set NEMOCLAW_SANDBOX_NAME in the environment to override."
        ) from exc


def _detect_sandbox_name(payload: dict) -> str:
    default = (payload.get("defaultSandbox") or "").strip()
    names = [s.get("name") for s in (payload.get("sandboxes") or []) if s.get("name")]
    if default:
        return default
    if len(names) == 1:
        return names[0]
    raise RuntimeError(
        "No default NemoClaw sandbox; set NEMOCLAW_SANDBOX_NAME or run "
        f"`nemoclaw use <name>` (known: {', '.join(map(repr, names)) or '(none)'})."
    )


NEMOCLAW_SANDBOX_NAME = os.environ.get("NEMOCLAW_SANDBOX_NAME", "").strip() or _detect_sandbox_name(
    _nemoclaw_list_payload()
)
print("Sandbox:", NEMOCLAW_SANDBOX_NAME)

ports = " ".join(str(port) for port in TEST_PORTS)
probe = f"""
HOST_IP="${{HOST_IP:-{HOST_INTERNAL_ALIAS}}}"
PORTS="{ports}"

echo __PORT_REACHABILITY_PROBE_BEGIN__
for port in $PORTS; do
  body="$(mktemp)"
  http_code="$(curl -sS --max-time 5 -o "$body" -w '%{{http_code}}' "http://${{HOST_IP}}:${{port}}/" 2>/tmp/portcheck.err)"
  curl_exit=$?
  body_text="$(tr '[:upper:]' '[:lower:]' < "$body")"

  if printf '%s' "$body_text" | grep -Eq 'policy_denied|egress.*denied|not allowed by policy'; then
    status="NOT_REACHABLE"
    note="blocked by policy"
  elif [ "$http_code" = "502" ] || printf '%s' "$body_text" | grep -q 'upstream_unreachable'; then
    status="NOT_REACHABLE"
    note="allowed, no service listening"
  elif [ "$http_code" != "000" ]; then
    status="REACHABLE"
    note="HTTP $http_code response"
  else
    status="NOT_REACHABLE"
    case "$curl_exit" in
      7) note="connect failed" ;;
      28) note="timeout" ;;
      *) note="curl_exit=$curl_exit" ;;
    esac
  fi

  printf '%s|%s|%s|%s\n' "$port" "$status" "$http_code" "$note"
  rm -f "$body" /tmp/portcheck.err
done
echo __PORT_REACHABILITY_PROBE_END__
exit
""".strip() + "\n"

CONNECT_CLI = "nemoclaw"
print(f"Running reachability probe through: {CONNECT_CLI} {NEMOCLAW_SANDBOX_NAME} connect")
result = subprocess.run(
    [CONNECT_CLI, NEMOCLAW_SANDBOX_NAME, "connect"],
    input=probe,
    capture_output=True,
    text=True,
    timeout=120,
)

lines = result.stdout.splitlines()
ansi_escape = re.compile(r"\x1b\[[0-?]*[ -/]*[@-~]")
try:
    begin = next(i for i, line in enumerate(lines) if line.strip() == "__PORT_REACHABILITY_PROBE_BEGIN__")
    end = next(i for i, line in enumerate(lines[begin + 1 :], start=begin + 1) if line.strip() == "__PORT_REACHABILITY_PROBE_END__")
    rows = []
    for line in lines[begin + 1 : end]:
        parts = ansi_escape.sub("", line).strip().split("|", 3)
        if len(parts) == 4 and parts[0].isdigit():
            rows.append(parts)

    print(f"{'PORT':>5}  {'STATUS':<13}  {'HTTP':>4}  NOTE")
    print(f"{'----':>5}  {'------':<13}  {'----':>4}  ----")
    for port, status, http_code, note in rows:
        print(f"{port:>5}  {status:<13}  {http_code:>4}  {note}")
except StopIteration:
    print(result.stdout)

if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(
        f"{CONNECT_CLI} {NEMOCLAW_SANDBOX_NAME} connect probe failed with exit code {result.returncode}"
    )
